# 🧹 04. Data Preprocessing Pipeline

This notebook implements basic dataset quality profiling, handles missing fields, eliminates duplicate records, Cap outliers using IQR bounds, and structures a reusable preprocessing pipeline class.

## 1. Project Introduction & Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Append project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data.load_data import DataLoader
from src.preprocessing.duplicates import DuplicateHandler
from src.preprocessing.missing_values import MissingValueImputer
from src.preprocessing.outliers import OutlierCapper

print("Preprocessing modules imported successfully.")

## 2. Load Dataset

In [ ]:
loader = DataLoader()
app_df, credit_df = loader.load_all()
print(f"Application Shape: {app_df.shape}")

## 3. Duplicate Records Analysis

In [ ]:
dup_handler = DuplicateHandler()
app_clean = dup_handler.remove_duplicates(app_df)
print(f"Cleaned Application Shape: {app_clean.shape}")

## 4. Invalid Data Corrections

In [ ]:
# Correct negative children, negative income, negative family size
app_clean['AMT_INCOME_TOTAL'] = app_clean['AMT_INCOME_TOTAL'].abs()
app_clean['CNT_CHILDREN'] = app_clean['CNT_CHILDREN'].clip(lower=0)
app_clean['CNT_FAM_MEMBERS'] = app_clean['CNT_FAM_MEMBERS'].clip(lower=1)
print("Invalid data boundary corrections executed.")

## 5. Missing Values Treatment

In [ ]:
imputer = MissingValueImputer()
numerical_cols = ["AMT_INCOME_TOTAL", "CNT_CHILDREN", "CNT_FAM_MEMBERS"]
categorical_cols = ["CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "OCCUPATION_TYPE"]

imputer.fit(app_clean, numerical_cols, categorical_cols)
app_imputed = imputer.transform(app_clean)
print(f"Null counts after imputation:\n{app_imputed.isnull().sum()}")

## 6. Outlier Capping

In [ ]:
capper = OutlierCapper(factor=1.5)
capper.fit(app_imputed, numerical_cols)
app_capped = capper.transform(app_imputed)
print("Outliers successfully capped.")

## 7. Pipeline Execution Preview

In [ ]:
from src.preprocessing.pipeline import PreprocessingPipeline

pipeline = PreprocessingPipeline()
train_shape, test_shape = pipeline.execute_full_pipeline()
print(f"Pipeline executed. Balanced Train shape: {train_shape}, Test shape: {test_shape}")

## 8. Conclusion
The preprocessing pipeline successfully cleaned illogical ranges, removed duplicates, imputed missing variables, capped outliers, and serialized preprocessing transformers. The balanced training set is now saved and ready for feature engineering.